In [1]:
import os
os.environ["TRAIN_MANIFEST"] = "/home/mohamed/Mohamed/Vodafone_project/all_eg_speech/train/train_abuelnasr_all_eg_speech_manifest.json"
os.environ["TEST_MANIFEST"] = "/home/mohamed/Mohamed/Vodafone_project/all_eg_speech/test/test_abuelnasr_all_eg_speech_manifest.json"

In [2]:
import torch
from omegaconf import OmegaConf, open_dict
from pytorch_lightning import Trainer

import nemo.collections.asr as nemo_asr

In [3]:
cfg = nemo_asr.models.ASRModel.restore_from("/home/mohamed/Mohamed/Vodafone_project/conformer_ar.nemo", return_config=True)

In [4]:
# cfg

In [5]:
from nemo.core import adapter_mixins

# Utility method to check and update the model config
def update_model_config_to_support_adapter(model_cfg):
    with open_dict(model_cfg):
        adapter_metadata = adapter_mixins.get_registered_adapter(model_cfg.encoder._target_)
        if adapter_metadata is not None:
            model_cfg.encoder._target_ = adapter_metadata.adapter_class_path
    
    print("Updated encoder _target_ model :", model_cfg.encoder._target_)
    return model_cfg

In [6]:
cfg = update_model_config_to_support_adapter(cfg)

In [7]:
model = nemo_asr.models.ASRModel.restore_from("/home/mohamed/Mohamed/Vodafone_project/projects/app/scripts/experiments/ASR-Adapters/2024-10-10_22-37-44/checkpoints/ASR-Adapters.nemo")

[NeMo I 2024-10-11 14:59:48 mixins:172] Tokenizer SentencePieceTokenizer initialized with 128 tokens


[NeMo W 2024-10-11 14:59:48 modelPT:165] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    batch_size: 6
    bucketing_batch_size: null
    bucketing_strategy: synced_randomized
    is_tarred: false
    manifest_filepath: /home/mohamed/Mohamed/Vodafone_project/all_eg_speech/train/train_abuelnasr_all_eg_speech_manifest.json
    max_duration: 30.0
    min_duration: 0.1
    num_workers: 16
    pin_memory: true
    sample_rate: 16000
    shuffle: true
    shuffle_n: 2048
    tarred_audio_filepaths: null
    trim_silence: false
    use_start_end_token: false
    
[NeMo W 2024-10-11 14:59:48 modelPT:172] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    batch_size:

[NeMo I 2024-10-11 14:59:48 features:289] PADDING: 0
[NeMo I 2024-10-11 14:59:50 adapter_mixins:612] Finished setup of adapter : 'AN4'. Enabled: True.
[NeMo I 2024-10-11 14:59:50 save_restore_connector:249] Model EncDecCTCModelBPE was successfully restored from /home/mohamed/Mohamed/Vodafone_project/projects/app/scripts/experiments/ASR-Adapters/2024-10-10_22-37-44/checkpoints/ASR-Adapters.nemo.


In [8]:
import wandb
wandb.login()

In [9]:
# from pytorch_lightning.loggers import WandbLogger

# wandb_logger = WandbLogger(project="ctc_ar_adapter_2")


In [10]:
# from pytorch_lightning.callbacks import ModelCheckpoint
# checkpoint_callback = ModelCheckpoint(dirpath="my/path/", save_top_k=2, monitor="val_loss")


In [11]:
accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'

trainer = Trainer(devices=1, accelerator=accelerator, max_epochs=20,
                  enable_checkpointing=False, enable_progress_bar=True, val_check_interval=225, logger=False)

model.set_trainer(trainer)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [12]:
# utility method
import json
from nemo.collections.asr.parts.utils.manifest_utils import read_manifest


In [13]:
# if not os.path.exists('scripts/transcribe_speech.py'):
#   !wget -P scripts/ https://raw.githubusercontent.com/NVIDIA/NeMo/main/examples/asr/transcribe_speech.py

# if not os.path.exists('scripts/speech_to_text_eval.py'):
#   !wget -P scripts/ https://raw.githubusercontent.com/NVIDIA/NeMo/main/examples/asr/speech_to_text_eval.py

In [14]:
# model.save_to("unadapted_model.nemo")

In [15]:
os.environ["HYDRA_FULL_ERROR"]="1"

In [16]:
# !python scripts/speech_to_text_eval.py \
#   model_path="/content/unadapted_model.nemo" \
#   dataset_manifest=$TEST_MANIFEST \
#   output_filename="/content/unadapted_predictions.json" \
#   batch_size=32 \
#   use_cer=False +map_location=gpu 

In [17]:
with open_dict(model.cfg):
  # Train Dataloader
  model.cfg.train_ds.manifest_filepath = os.environ["TRAIN_MANIFEST"]
  model.cfg.train_ds.batch_size = 6
  model.cfg.train_ds.is_tarred = False
  model.cfg.train_ds.tarred_audio_filepaths = None

  model.cfg.validation_ds.manifest_filepath = os.environ["TEST_MANIFEST"]
  model.cfg.validation_ds.batch_size = 6

model.setup_training_data(model.cfg.train_ds)
model.setup_multiple_validation_data(model.cfg.validation_ds)
# model.setup_multiple_test_data(model.cfg.validation_ds)

[NeMo I 2024-10-11 14:59:52 collections:196] Dataset loaded with 7628 files totalling 11.07 hours
[NeMo I 2024-10-11 14:59:52 collections:197] 0 files were filtered totalling 0.00 hours


[NeMo W 2024-10-11 14:59:52 nemo_logging:349] /home/mohamed/miniconda3/envs/torch/lib/python3.10/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
      warnings.warn(_create_warning_msg(
    


[NeMo I 2024-10-11 14:59:52 collections:196] Dataset loaded with 405 files totalling 0.57 hours
[NeMo I 2024-10-11 14:59:52 collections:197] 0 files were filtered totalling 0.00 hours


In [18]:
with open_dict(model.cfg):
  # Spec Augment
  model.cfg.spec_augment.freq_masks = model.cfg.spec_augment.freq_masks  # Can be changed
  model.cfg.spec_augment.freq_width = model.cfg.spec_augment.freq_width  # Can be changed
  model.cfg.spec_augment.time_masks = model.cfg.spec_augment.time_masks  # Can be changed
  model.cfg.spec_augment.time_width = model.cfg.spec_augment.time_width  # Can be changed

model.spec_augmentation = model.from_config_dict(model.cfg.spec_augment)

In [19]:
if 'optim' in model.cfg:
  print(OmegaConf.to_yaml(model.cfg.optim))

betas:
- 0.9
- 0.98
lr: 0.1
name: adamw
sched:
  d_model: 512
  min_lr: 1.0e-06
  name: NoamAnnealing
  warmup_ratio: null
  warmup_steps: 100
weight_decay: 0.0



In [20]:
with open_dict(model.cfg):
  model.cfg.optim.lr = 0.01
  model.cfg.optim.weight_decay = 0.0001
  model.cfg.optim.sched.warmup_steps = 600

model.setup_optimization(model.cfg.optim)

[NeMo I 2024-10-11 14:59:52 modelPT:723] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: [0.9, 0.98]
        capturable: False
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 0.01
        maximize: False
        weight_decay: 0.0001
    )
[NeMo I 2024-10-11 14:59:52 lr_scheduler:915] Scheduler "<nemo.core.optim.lr_scheduler.NoamAnnealing object at 0x7f3aaeb89480>" 
    will be used during training (effective maximum steps = 25440) - 
    Parameters : 
    (d_model: 512
    min_lr: 1.0e-06
    warmup_ratio: null
    warmup_steps: 600
    max_steps: 25440
    )


(AdamW (
 Parameter Group 0
     amsgrad: False
     betas: [0.9, 0.98]
     capturable: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     initial_lr: 0.01
     lr: 3.0070326520293014e-08
     maximize: False
     weight_decay: 0.0001
 ),
 {'scheduler': <nemo.core.optim.lr_scheduler.NoamAnnealing at 0x7f3aaeb89480>,
  'interval': 'step',
  'frequency': 1,
  'monitor': 'loss',
  'reduce_on_plateau': False})

In [21]:
if hasattr(model, 'adapter_module_names'):
  print(model.adapter_module_names)

['', 'encoder', 'decoder', 'joint']


In [22]:
for module in model.children():
  if hasattr(module, 'get_accepted_adapter_types'):
    types = module.get_accepted_adapter_types()
    print("Module : ", module.__class__.__name__)

    for tp in types:
      print(tp)
    print()

Module :  ConformerEncoderAdapter
<class 'nemo.collections.common.parts.adapter_modules.LinearAdapter'>
<class 'nemo.collections.asr.parts.submodules.adapters.multi_head_attention_adapter_module.MultiHeadAttentionAdapter'>
<class 'nemo.collections.asr.parts.submodules.adapters.multi_head_attention_adapter_module.RelPositionMultiHeadAttentionAdapter'>

Module :  ConvASRDecoder
<class 'nemo.collections.common.parts.adapter_modules.LinearAdapter'>



In [23]:
from nemo.collections.common.parts.adapter_modules import LinearAdapterConfig

In [24]:
#%% [code]
#@title Adapter Setup { display-mode: "form" }
adapter_name = "AN4" #@param {type:"string"}
adapter_dim = 32 #@param {type:"integer"}
adapter_activation = "swish" #@param {type:"string"}
adapter_norm_position = "pre" #@param ["pre", "post"]

In [25]:
adapter_cfg = LinearAdapterConfig(
    in_features=model.cfg.encoder.d_model,  # conformer specific model dim. Every layer emits this dim at its output.
    dim=adapter_dim,  # the bottleneck dimension of the adapter
    activation=adapter_activation,  # activation used in bottleneck block
    norm_position=adapter_norm_position,  # whether to use LayerNorm at the beginning or the end of the adapter
)
print(adapter_cfg)

In [26]:
model.summarize()

  | Name              | Type                              | Params
------------------------------------------------------------------------
0 | preprocessor      | AudioToMelSpectrogramPreprocessor | 0     
1 | encoder           | ConformerEncoderAdapter           | 122 M 
2 | decoder           | ConvASRDecoder                    | 66.2 K
3 | loss              | CTCLoss                           | 0     
4 | spec_augmentation | SpectrogramAugmentation           | 0     
5 | wer               | WER                               | 0     
------------------------------------------------------------------------
122 M     Trainable params
0         Non-trainable params
122 M     Total params
488.438   Total estimated model params size (MB)

In [27]:
model.add_adapter(name=adapter_name, cfg=adapter_cfg)

In [28]:
model.summarize()

  | Name              | Type                              | Params
------------------------------------------------------------------------
0 | preprocessor      | AudioToMelSpectrogramPreprocessor | 0     
1 | encoder           | ConformerEncoderAdapter           | 122 M 
2 | decoder           | ConvASRDecoder                    | 66.2 K
3 | loss              | CTCLoss                           | 0     
4 | spec_augmentation | SpectrogramAugmentation           | 0     
5 | wer               | WER                               | 0     
------------------------------------------------------------------------
122 M     Trainable params
0         Non-trainable params
122 M     Total params
488.438   Total estimated model params size (MB)

In [29]:
model.set_enabled_adapters(enabled=False)  # disable all adapters
model.set_enabled_adapters(name=adapter_name, enabled=True)  # enable only the current adapter we want to train

[NeMo I 2024-10-11 14:59:52 adapter_mixins:719] Setting adapter 'AN4' status : Enabled = False
[NeMo I 2024-10-11 14:59:52 adapter_mixins:734] Setting adapter 'AN4' status : Enabled = True


In [30]:
model.freeze()
model.unfreeze_enabled_adapters()

[NeMo I 2024-10-11 14:59:52 adapter_mixins:405] Froze module encoder.layers.0.conv.batch_norm: BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2024-10-11 14:59:52 adapter_mixins:405] Froze module encoder.layers.1.conv.batch_norm: BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2024-10-11 14:59:52 adapter_mixins:405] Froze module encoder.layers.2.conv.batch_norm: BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2024-10-11 14:59:52 adapter_mixins:405] Froze module encoder.layers.3.conv.batch_norm: BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2024-10-11 14:59:52 adapter_mixins:405] Froze module encoder.layers.4.conv.batch_norm: BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2024-10-11 14:59:52 adapter_mixins:405] Froze module encoder.layers.5.conv.batch_norm: BatchNorm1d(512, eps

In [31]:
model.summarize()

  | Name              | Type                              | Params
------------------------------------------------------------------------
0 | preprocessor      | AudioToMelSpectrogramPreprocessor | 0     
1 | encoder           | ConformerEncoderAdapter           | 122 M 
2 | decoder           | ConvASRDecoder                    | 66.2 K
3 | loss              | CTCLoss                           | 0     
4 | spec_augmentation | SpectrogramAugmentation           | 0     
5 | wer               | WER                               | 0     
------------------------------------------------------------------------
608 K     Trainable params
121 M     Non-trainable params
122 M     Total params
488.438   Total estimated model params size (MB)

In [32]:
trainer.logger

In [33]:
# Prepare NeMo's Experiment manager to handle checkpoint saving and logging for us
from nemo.utils import exp_manager

# Environment variable generally used for multi-node multi-gpu training.
# In notebook environments, this flag is unnecessary and can cause logs of multiple training runs to overwrite each other.
os.environ.pop('NEMO_EXPM_VERSION', None)


exp_config = exp_manager.ExpManagerConfig(
    exp_dir=f'experiments/',
    name=f"ASR-Adapters",
    create_wandb_logger=True,
    wandb_logger_kwargs={"project":"ctc_ar_adapter_2"},
    checkpoint_callback_params=exp_manager.CallbackParams(
        monitor="val_wer",
        mode="min",
        always_save_nemo=True,
        save_best_model=True,
    ),
)

exp_config = OmegaConf.structured(exp_config)

logdir = exp_manager.exp_manager(trainer, exp_config)

[NeMo I 2024-10-11 14:59:52 exp_manager:396] Experiments will be logged at experiments/ASR-Adapters/2024-10-11_14-59-52
[NeMo I 2024-10-11 14:59:52 exp_manager:842] TensorboardLogger has been set up


Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: abuelnasr. Use `wandb login --relogin` to force relogin


[NeMo I 2024-10-11 15:00:11 exp_manager:857] WandBLogger has been set up


In [34]:
trainer.fit(model)

2024-10-11 15:00:12.743821: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-11 15:00:12.743878: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-11 15:00:12.795480: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-11 15:00:12.913796: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-11 15:00:14.318600: W tensorflow/compiler/tf2

[NeMo I 2024-10-11 15:00:15 modelPT:723] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: [0.9, 0.98]
        capturable: False
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 0.01
        maximize: False
        weight_decay: 0.0001
    )
[NeMo I 2024-10-11 15:00:15 lr_scheduler:915] Scheduler "<nemo.core.optim.lr_scheduler.NoamAnnealing object at 0x7f39f57d8100>" 
    will be used during training (effective maximum steps = 25440) - 
    Parameters : 
    (d_model: 512
    min_lr: 1.0e-06
    warmup_ratio: null
    warmup_steps: 600
    max_steps: 25440
    )



  | Name              | Type                              | Params
------------------------------------------------------------------------
0 | preprocessor      | AudioToMelSpectrogramPreprocessor | 0     
1 | encoder           | ConformerEncoderAdapter           | 122 M 
2 | decoder           | ConvASRDecoder                    | 66.2 K
3 | loss              | CTCLoss                           | 0     
4 | spec_augmentation | SpectrogramAugmentation           | 0     
5 | wer               | WER                               | 0     
------------------------------------------------------------------------
608 K     Trainable params
121 M     Non-trainable params
122 M     Total params
488.438   Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

[NeMo W 2024-10-11 15:00:15 nemo_logging:349] /home/mohamed/miniconda3/envs/torch/lib/python3.10/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
      warnings.warn(_create_warning_msg(
    


...

[NeMo I 2024-10-11 15:14:04 wer:318] 
    
[NeMo I 2024-10-11 15:14:04 wer:319] reference:لل للاسره والمجتمع اللي حواليه بصي بقي انا عايز قبل ما نبدا بالحته دي اشوف بقي الولد ده وهو بيذاكر يا شيماء الفيديو التاني لو جاهز نشوفه وهو بيذاكر
[NeMo I 2024-10-11 15:14:04 wer:320] predicted:لل للاسره والمجتمع اللي حووليه بص بقي انا عايز قبل ما نبدا بالحته دي اشوف بقي الولد ده هو بيذاكر عشان الفيديوتاني لو جاهز نشوفه وهو بيذاكر
[NeMo I 2024-10-11 15:14:04 wer:318] 
    
[NeMo I 2024-10-11 15:14:04 wer:319] reference:آه ظهر بشكل كبير جداً في فترة الأخيرة
[NeMo I 2024-10-11 15:14:04 wer:320] predicted:ا ظهر بشكل كبير جدا في الفتره الأخيرة
[NeMo I 2024-10-11 15:14:05 wer:318] 
    
[NeMo I 2024-10-11 15:14:05 wer:319] reference:لم يكونوا من الإخوان المسلمين كما يتحدثون
[NeMo I 2024-10-11 15:14:05 wer:320] predicted:لم يكونوا من الاخوان المسلمين كما يتحدثون
[NeMo I 2024-10-11 15:14:05 wer:318] 
    
[NeMo I 2024-10-11 15:14:05 wer:319] reference:عامل إيه واخبارك إيه طمني عنك
[NeMo I 2024-10-11 15:

Epoch 0, global step 900: 'val_wer' reached 0.13268 (best 0.12937), saving model to 'experiments/ASR-Adapters/2024-10-11_14-59-52/checkpoints/ASR-Adapters--val_wer=0.1327-epoch=0.ckpt' as top 3


[NeMo I 2024-10-11 15:14:56 wer:318] 
    
[NeMo I 2024-10-11 15:14:56 wer:319] reference:والمواقف الغير رسمية
[NeMo I 2024-10-11 15:14:56 wer:320] predicted:والمواقف الغير رسمية
[NeMo I 2024-10-11 15:15:25 wer:318] 
    
[NeMo I 2024-10-11 15:15:25 wer:319] reference:في في منتهي الجمال هل الشوز نفس الحكايه ولا دي ايه بقي لاء هو كمان نفس اللا نفس المصنع ا ه نفس المصنع اللي بيعمل لي شنط هو اللي بيعمل لي شوز شوفوا يا بنات
[NeMo I 2024-10-11 15:15:25 wer:320] predicted:في في مستهل الجمال هلشوف نفس الحكايه ولا دي ل هو كمان نفس ال نفس المصن يعني نفس المصنع اللي بيعملي شنطه ولا بيعملي شن البنات
[NeMo I 2024-10-11 15:15:54 wer:318] 
    
[NeMo I 2024-10-11 15:15:54 wer:319] reference:لكن ما ينفعش ان انا اكل فيها ما ينفعش استعملها في المطبخ هي بس فقط للاستعمال الخارجي
[NeMo I 2024-10-11 15:15:54 wer:320] predicted:لكن ما ينفعش ان انا اكل فيها ما ينفعش استخملها في المطبخ هيبث فقط الاستعمال الخارجي


[NeMo W 2024-10-11 15:16:22 nemo_logging:349] /home/mohamed/miniconda3/envs/torch/lib/python3.10/site-packages/pytorch_lightning/trainer/call.py:53: UserWarning: Detected KeyboardInterrupt, attempting graceful shutdown...
      rank_zero_warn("Detected KeyboardInterrupt, attempting graceful shutdown...")
    


wandb: Network error (ConnectTimeout), entering retry loop.
wandb: Network error resolved after 0:01:23.900246, resuming normal operation.
wandb: Network error (ConnectionError), entering retry loop.
wandb: Network error resolved after 0:02:39.490250, resuming normal operation.
